# ST-HGRec pipeline — SVKM AI/ML cluster (Jupyter)

Upload this notebook via **Files → Upload** into a Jupyter job on Altair Access
(Container Image: `pytorch_pbs:23.06-py3`, GPUs per Node: `1`), then run the
cells top to bottom.

Each `%%bash` cell calls a venv's binaries by full path (`env-rec/bin/python`,
`env-absa/bin/python`) instead of `source .../activate` — activation doesn't
persist across notebook cells since each one can run as its own subshell.

**Edit the `CLUSTER_USER` cell below before running anything else.**

Stages: clone → build env-rec → raw data + atomic files → build env-absa →
ABSA extraction (the long one; safe to interrupt and re-run) → normalize /
trust / graph → baselines → ST-HGRec + ablations → copy results to
persistent storage outside this job's working directory.

In [ ]:
!nvidia-smi

## Config — edit `CLUSTER_USER` to your Altair Access username

In [ ]:
CLUSTER_USER = "CHANGE_ME"  # <-- your Altair Access username (top-right of the portal)

import os, datetime

assert CLUSTER_USER != "CHANGE_ME", "Set CLUSTER_USER above before continuing."

REPO_ROOT = f"/data/{CLUSTER_USER}/sentimentgraph"
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_DIR = f"/data/{CLUSTER_USER}/st-hgrec-results/{RUN_ID}"
os.makedirs(os.path.dirname(RESULTS_DIR), exist_ok=True)

print("REPO_ROOT   :", REPO_ROOT)
print("RUN_ID      :", RUN_ID)
print("RESULTS_DIR :", RESULTS_DIR)

## 0. Clone the repo (skips cleanly if already cloned)

In [ ]:
import subprocess

if not os.path.isdir(REPO_ROOT):
    subprocess.run(
        ["git", "clone", "https://github.com/devanshi1308/sentimentgraph.git", REPO_ROOT],
        check=True,
    )
else:
    print("Already cloned.")

os.chdir(REPO_ROOT)
print("cwd:", os.getcwd())

## 1. env-rec — build once (skipped automatically on later reruns)

In [ ]:
%%bash
if [ ! -d env-rec ]; then
    python -m venv env-rec
    env-rec/bin/pip install --upgrade pip
    env-rec/bin/pip install -r environments/requirements-rec.txt
    env-rec/bin/pip install --force-reinstall torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
fi
env-rec/bin/python -c "import torch; print('CUDA available:', torch.cuda.is_available())"

## 2. Raw data + atomic files (env-rec)

In [ ]:
%%bash
env-rec/bin/python -m src.data.download
env-rec/bin/python -m src.data.build_atomic

## 3. env-absa — build once, separate from env-rec (never mix)

In [ ]:
%%bash
if [ ! -d env-absa ]; then
    python -m venv env-absa
    env-absa/bin/pip install --upgrade pip
    env-absa/bin/pip install -r environments/requirements-absa.txt
    env-absa/bin/pip install --force-reinstall torch==2.3.1 --index-url https://download.pytorch.org/whl/cu121
    env-absa/bin/python -m spacy download en_core_web_sm
fi
env-absa/bin/python -c "import torch; print('CUDA available:', torch.cuda.is_available())"

## 4. ABSA extraction — the long one

Checkpointed against `data/interim/absa_full_progress.txt`. If this cell
times out, stops, or the kernel dies, just re-run this same cell — it
resumes where it left off rather than starting over.

In [ ]:
%%bash
env-absa/bin/python -m src.sentiment.run_absa --device auto --batch-size 256

## 5. Normalize sentiment, trust signals, graph (env-rec)

In [ ]:
%%bash
env-rec/bin/python -m src.sentiment.normalize_aspects
env-rec/bin/python -m src.data.trust_signals
env-rec/bin/python -m src.graph.build_graph

## 6. Baselines (Popularity, BPR, LightGCN, NGCF)

In [ ]:
%%bash
env-rec/bin/python -m src.eval.run_baselines --config configs/popularity.yaml
env-rec/bin/python -m src.eval.run_baselines --config configs/bpr.yaml
env-rec/bin/python -m src.eval.run_baselines --config configs/lightgcn.yaml
env-rec/bin/python -m src.eval.run_baselines --config configs/ngcf.yaml

## 7. ST-HGRec — determinism check, then the full run + ablations

In [ ]:
%%bash
env-rec/bin/python -m src.eval.run_sthgrec --config configs/sthgrec.yaml --check-determinism

In [ ]:
%%bash
env-rec/bin/python -m src.eval.run_sthgrec --config configs/sthgrec.yaml
env-rec/bin/python -m src.eval.ablations --config configs/sthgrec.yaml

## 8. Copy results to persistent storage

This job's own working directory is ephemeral — this is the step that
actually guarantees the results survive after the job ends.

In [ ]:
import shutil

os.makedirs(RESULTS_DIR, exist_ok=True)
shutil.copytree("results", os.path.join(RESULTS_DIR, "results"), dirs_exist_ok=True)
for f in ("data/interim/absa_normalized.csv", "data/interim/trust_signals.csv"):
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)

print("Saved to:", RESULTS_DIR)
!ls -la "$RESULTS_DIR"